Generate Forecasts

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

n_periods = 252 # Forecasting for approximately one year
forecast_result = auto_model.get_forecast(steps=n_periods)
forecast = forecast_result.predicted_mean
confidence_intervals = forecast_result.conf_int()

# Create a date range for the forecast period
forecast_index = pd.date_range(start=tsla_data.index[-1] + pd.Timedelta(days=1), periods=n_periods, freq='B')
forecast_series = pd.Series(forecast, index=forecast_index)
confidence_intervals.index = forecast_index

# Plot the forecast
plt.figure(figsize=(15, 8))
plt.plot(tsla_data['Adj Close']['2023-01-01':], label='Historical Prices')
plt.plot(forecast_series, label='Forecast', color='red')
plt.fill_between(confidence_intervals.index,
                 confidence_intervals.iloc[:, 0],
                 confidence_intervals.iloc[:, 1], color='pink', alpha=0.5, label='95% Confidence Interval')
plt.title('TSLA Stock Price Forecast (Next 12 Months)')
plt.xlabel('Date')
plt.ylabel('Adjusted Close Price (USD)')
plt.legend()
plt.grid(True)
plt.savefig('../results/plots/tsla_forecast.png')
plt.show()

Calculate MPT

In [ ]:
import numpy as np

# 1. Expected Returns
# For TSLA, use the 1-year forecast
last_price_tsla = tsla_data['Adj Close'][-1]
forecasted_price_tsla = forecast_series[-1]
expected_return_tsla = (forecasted_price_tsla - last_price_tsla) / last_price_tsla

# For BND and SPY, use historical annualized returns
daily_returns = data['Adj Close'].pct_change().dropna()
annual_returns = daily_returns.mean() * 252
expected_return_bnd = annual_returns['BND']
expected_return_spy = annual_returns['SPY']

expected_returns_vector = np.array([expected_return_tsla, expected_return_bnd, expected_return_spy])
print(f"Annualized Expected Returns:\nTSLA: {expected_return_tsla:.2%}\nBND: {expected_return_bnd:.2%}\nSPY: {expected_return_spy:.2%}")

# 2. Covariance Matrix
cov_matrix_annual = daily_returns[['TSLA', 'BND', 'SPY']].cov() * 252
print("\nAnnual Covariance Matrix:\n", cov_matrix_annual)

Monte Carlo Simulation

In [ ]:
# Monte Carlo Simulation
num_portfolios = 25000
results = np.zeros((3, num_portfolios))
weights_record = []
risk_free_rate = 0.02 # Assumption for Sharpe Ratio calculation

for i in range(num_portfolios):
    # Random weights
    weights = np.random.random(3)
    weights /= np.sum(weights)
    weights_record.append(weights)

    # Portfolio return and volatility
    portfolio_return = np.sum(weights * expected_returns_vector)
    portfolio_stddev = np.sqrt(np.dot(weights.T, np.dot(cov_matrix_annual, weights))) # [15, 18, 20]

    results[0,i] = portfolio_return
    results[1,i] = portfolio_stddev
    # Sharpe Ratio
    results[2,i] = (portfolio_return - risk_free_rate) / portfolio_stddev # [5, 13]

# Convert results to a DataFrame
results_df = pd.DataFrame(results.T, columns=['Return', 'Volatility', 'Sharpe Ratio'])
results_df['TSLA_Weight'] = [w[0] for w in weights_record]
results_df['BND_Weight'] = [w[1] for w in weights_record]
results_df['SPY_Weight'] = [w[2] for w in weights_record]

# Identify key portfolios
max_sharpe_portfolio = results_df.iloc[results_df['Sharpe Ratio'].idxmax()]
min_vol_portfolio = results_df.iloc[results_df['Volatility'].idxmin()]

print("Maximum Sharpe Ratio Portfolio:\n", max_sharpe_portfolio)
print("\nMinimum Volatility Portfolio:\n", min_vol_portfolio)

Visualize Frontier

In [ ]:
# Plot the Efficient Frontier
plt.figure(figsize=(12, 8))
plt.scatter(results_df['Volatility'], results_df['Return'], c=results_df['Sharpe Ratio'], cmap='viridis', marker='o')
plt.colorbar(label='Sharpe Ratio')
plt.title('Efficient Frontier of Portfolios (TSLA, BND, SPY)')
plt.xlabel('Annualized Volatility (Risk)')
plt.ylabel('Annualized Expected Return')

# Mark the key portfolios
plt.scatter(max_sharpe_portfolio['Volatility'], max_sharpe_portfolio['Return'], color='red', marker='*', s=300, label='Maximum Sharpe Ratio')
plt.scatter(min_vol_portfolio['Volatility'], min_vol_portfolio['Return'], color='blue', marker='*', s=300, label='Minimum Volatility')

plt.legend(labelspacing=0.8)
plt.grid(True)
plt.savefig('../results/plots/efficient_frontier.png')
plt.show()

BackTesting

In [ ]:
# Define backtesting period
backtest_start = '2024-08-01'
backtest_end = '2025-07-31'
backtest_data = daily_returns[backtest_start:backtest_end]

# Get optimal and benchmark weights
strategy_weights = max_sharpe_portfolio[['TSLA_Weight', 'BND_Weight', 'SPY_Weight']].values
benchmark_weights = np.array([0, 0.40, 0.60]) # TSLA, BND, SPY order

# Calculate portfolio returns
strategy_returns = backtest_data[['TSLA', 'BND', 'SPY']].dot(strategy_weights)
benchmark_returns = backtest_data[['TSLA', 'BND', 'SPY']].dot(benchmark_weights)

# Calculate cumulative returns
strategy_cumulative_returns = (1 + strategy_returns).cumprod()
benchmark_cumulative_returns = (1 + benchmark_returns).cumprod()

# Plot the results
plt.figure(figsize=(14, 7))
plt.plot(strategy_cumulative_returns, label='Model-Driven Strategy')
plt.plot(benchmark_cumulative_returns, label='60/40 Benchmark')
plt.title('Backtest: Strategy vs. Benchmark Cumulative Returns')
plt.xlabel('Date')
plt.ylabel('Cumulative Returns')
plt.legend()
plt.grid(True)
plt.savefig('../results/plots/backtest_results.png')
plt.show()

# Calculate final performance metrics
total_return_strategy = strategy_cumulative_returns[-1] - 1
total_return_benchmark = benchmark_cumulative_returns[-1] - 1

# Annualize Sharpe Ratio [11, 16, 19]
sharpe_ratio_strategy = (strategy_returns.mean() * 252) / (strategy_returns.std() * np.sqrt(252))
sharpe_ratio_benchmark = (benchmark_returns.mean() * 252) / (benchmark_returns.std() * np.sqrt(252))

print(f"Strategy Total Return: {total_return_strategy:.2%}")
print(f"Benchmark Total Return: {total_return_benchmark:.2%}")
print(f"Strategy Sharpe Ratio: {sharpe_ratio_strategy:.2f}")
print(f"Benchmark Sharpe Ratio: {sharpe_ratio_benchmark:.2f}")